# 02 — Data Quality Check
## Masterproef: Spatiotemporal Prediction and Optimization of Car Parking in Mechelen

---

### Doel van dit notebook

Dit notebook voert een systematische datakwaliteitscontrole uit op de drie 
kernbestanden:
- `timeseries_shortterm.parquet` (697.218 rijen, 2018–2026, passanten)
- `timeseries_longterm.parquet`  (46.643 rijen, 2024 only, abonnees)
- `parking_location.parquet`     (15 parkings, locatie + capaciteit)

Het eindresultaat zijn twee gecleande tussenbestanden:
- `data_intermediate/shortterm_cleaned.parquet`
- `data_intermediate/longterm_cleaned.parquet`

Én een kwaliteitsrapport: `data_intermediate/quality_report.csv`

---

### Academische onderbouwing

Datakwaliteit is een kritische maar vaak onderschatte stap in het opbouwen van 
betrouwbare voorspellingsmodellen. Tanui et al. (2025) tonen aan dat outlier-handling 
via ROLS (Recursive OLS) de modelnauwkeurigheid met tot 12% verbetert ten opzichte 
van ruwe data. Nezhadettehad et al. (2025) benadrukken dat sensoruitval en 
data-artefacten in parkeerdata systemisch zijn en leiden tot geflatteerde 
prestatiecijfers als ze niet worden gedetecteerd.

Specifiek voor tijdreeksdata van parkeersensoren zijn drie types anomalieën 
gedocumenteerd (Gao et al., 2023; Wan et al., 2023):
1. **Frozen sensor values**: de sensor rapporteert herhaaldelijk exact dezelfde 
   bezettingswaarde over meerdere opeenvolgende uren → wijst op sensor-uitval.
2. **Physically impossible values**: `occupancy_rate > 1.0` of 
   `occupied_spaces > capacity` → wijst op registratiefout of capaciteitswijziging.
3. **Temporal gaps**: ontbrekende uurobservaties per parking → problematisch voor 
   lag-features en autocorrelatie-modellen (LSTM, SARIMA).

We documenteren elke anomalie in een kwaliteitsrapport zodat beslissingen 
transparant en reproduceerbaar zijn — een vereiste voor academische validiteit.

---

### Input / Output

| # | Bestand | Richting | Locatie |
|---|---|---|---|
| 1 | `timeseries_shortterm.parquet` | input | `data_raw/` |
| 2 | `timeseries_longterm.parquet`  | input | `data_raw/` |
| 3 | `parking_location.parquet`     | input | `data_raw/` |
| 4 | `shortterm_cleaned.parquet`    | output | `data_intermediate/` |
| 5 | `longterm_cleaned.parquet`     | output | `data_intermediate/` |
| 6 | `quality_report.csv`           | output | `data_intermediate/` |


In [1]:
import sys
print(sys.executable)



/Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/.venv/bin/python


In [2]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.parent.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.project_config import get_project_paths
from src.io.paths import resolve_data_path
from src.io.parking_readers import (
    read_longterm_timeseries,
    read_parking_location,
    read_shortterm_timeseries,
)
from src.phase1.quality import (
    FREEZE_THRESHOLD,
    KNOWN_CAPACITIES,
    QualityLogger,
    add_frozen_sensor_flag,
    apply_shortterm_quality_flags,
    check_dst_hour_anomalies,
    compute_shortterm_completeness,
    crosscheck_override_capacity,
    deduplicate_shortterm,
    evaluate_parking_location,
    process_longterm_quality,
    recompute_occupancy_for_inconsistent,
)

paths = get_project_paths(PROJECT_ROOT)
DATA_RAW = paths.data_raw
DATA_INT = paths.data_intermediate

qlogger = QualityLogger()
log = qlogger.log

print("Setup klaar.")



Setup klaar.


## Stap 1 — `parking_location.parquet` laden & inspecteren

De locatietabel is de referentie voor parking-identifiers, coördinaten, 
totale capaciteit en categorie (Centrum / Vesten / Rand / fiets).

Bekende metadata uit het schema:
- 15 rijen (11 auto + 4 fiets)
- 4 nulls bij `total_capacity` en `parking_location_category` → vermoedelijk 
  de 4 fietsparkings waarvoor capaciteit niet geregistreerd is
- `created_at` is VARCHAR (niet timestamp) → enkel voor referentie, niet voor join


In [3]:
df_loc = read_parking_location(PROJECT_ROOT)

print(f"Shape: {df_loc.shape}")
print(f"\nDatatypes:\n{df_loc.dtypes}")
print(f"\nNull-tellingen:\n{df_loc.isnull().sum()}")
print(f"\nData:\n{df_loc.to_string()}")



Shape: (15, 6)

Datatypes:
parking_id                        str
longitude                     float64
latitude                      float64
created_at                        str
total_capacity                float64
parking_location_category    category
dtype: object

Null-tellingen:
parking_id                   0
longitude                    0
latitude                     0
created_at                   0
total_capacity               4
parking_location_category    4
dtype: int64

Data:
                           parking_id  longitude   latitude                   created_at  total_capacity parking_location_category
0        Bike Halte Mechelen-Veemarkt   4.484340  51.029300  2024-08-29T12:54:57.5430000             NaN                       NaN
1               Bike Station Mechelen   4.483001  51.017822  2024-08-29T12:54:57.5430000             NaN                       NaN
2    Bike Station Mechelen (Arsenaal)   4.483918  51.016306  2024-08-29T12:54:57.5430000             NaN           

In [4]:
n_parkings, category_counts, capacity_check = evaluate_parking_location(
    df_loc,
    expected_n_parkings=15,
    known_capacities=KNOWN_CAPACITIES,
)
log("Aantal parkings", "parking_location", f"{n_parkings} unieke IDs", 0, "OK")

print("\nCategorie-verdeling:")
print(category_counts)

print("\nCapaciteitscheck vs. bekende waarden:")
print(capacity_check.to_string(index=False))

print("\nOK: parking_location geladen en gevalideerd.")




Categorie-verdeling:
parking_location_category
centrum    5
vesten     5
NaN        4
rand       1
Name: count, dtype: int64

Capaciteitscheck vs. bekende waarden:
     parking_id status  expected_capacity  actual_capacity
  P Grote Markt     OK                155              155
   P Hoogstraat     OK                109              109
   P Kathedraal     OK                130              130
        P Lamot     OK                255              255
     P Veemarkt     OK                129              129
        P Bruul     OK                350              350
        P Komet     OK                124              124
      P Maarten     OK                189              189
        P Tinel     OK                124              124
P Zandpoortvest     OK                622              622
      P Keerdok     OK                516              516

OK: parking_location geladen en gevalideerd.


## Stap 2 — `timeseries_shortterm.parquet` laden & basisinspectie

Bekend uit het schema:
- 697.218 rijen, 10 unieke parkings (enkel autoparkings)
- Tijdsbereik: `year` min=2018, max=2026 (maar 9 distinct years → check welke jaren)
- `publication_time`: TIMESTAMP_NS (nanoseconden, timezone-naïef)
- `created_at`: TIMESTAMP WITH TIME ZONE
- `occupancy_rate`: min=0, max=1 → **geen fysisch onmogelijke waarden op schema-niveau**
- `number_of_spaces_override`: min=50, max=416 (dynamische capaciteitsoverride)
- `number_of_occupied_spaces`: min=0, max=416


Let op: (voor in thesis) 
"P Zandpoortvest (622 plaatsen, vestentier) is opgenomen in de locatietabel maar beschikt over geen tijdreeksdata in de brondata van de Stad Mechelen. De vestentier is bijgevolg vertegenwoordigd door vier parkings (P Bruul, P Komet, P Maarten, P Tinel) met een gecombineerde capaciteit van 787 plaatsen, tegenover een werkelijke vestencapaciteit van 1.409 plaatsen inclusief P Zandpoortvest."


Aandachtspunt: `number_of_spaces_override` verschilt van `total_capacity` in 
`parking_location` omdat het de operationele capaciteit op dat tijdstip reflecteert 
(bijv. tijdelijke sluitingen van secties).



In [5]:
df_st = read_shortterm_timeseries(PROJECT_ROOT)

print(f"Shape         : {df_st.shape}")
print(f"Geheugen      : {df_st.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nDatatypes:\n{df_st.dtypes}")
print(f"\nNull-tellingen:\n{df_st.isnull().sum()}")



Shape         : (697218, 18)
Geheugen      : 139.6 MB

Datatypes:
parking_id                                          category
parking_id_hash                                     category
parking_type                                             str
kind                                                     str
publication_time                              datetime64[ns]
created_at                   datetime64[us, Europe/Brussels]
year                                                   int32
month                                                  int32
date_only                                             object
date_with_day                                            str
rounded_hour                                  datetime64[ns]
hour                                                   int32
weekday_int                                            int32
weekday_name                                             str
day_type                                                 str
number_of_spaces_ov

In [6]:
print("=== Jaarlijkse rij-telling (shortterm) ===")
print(df_st["year"].value_counts().sort_index())

print("\n=== Unieke parkings (shortterm) ===")
print(df_st["parking_id"].value_counts())

print("\n=== Datumbereik ===")
print(f"Min date_only : {df_st['date_only'].min()}")
print(f"Max date_only : {df_st['date_only'].max()}")

coverage = (
    df_st.groupby("parking_id")["date_only"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "start", "max": "end", "count": "n_rows"})
)
print("\n=== Dekking per parking ===")
print(coverage.to_string())



=== Jaarlijkse rij-telling (shortterm) ===
year
2018         5
2019     33000
2020     32741
2021     13630
2022     12762
2023     39980
2024    176280
2025    358280
2026     30540
Name: count, dtype: int64

=== Unieke parkings (shortterm) ===
parking_id
P Bruul                              80769
P Tinel                              79118
P Grote Markt                        77708
P Veemarkt                           77033
P Lamot                              76311
P Keerdok                            67446
P Hoogstraat                         64149
P Kathedraal                         63224
P Maarten                            57139
P Komet                              54321
Bike Halte Mechelen-Veemarkt             0
Bike Station Mechelen                    0
Bike Station Mechelen (Arsenaal)         0
Bike Station Mechelen-Nekkerspoel        0
P Zandpoortvest                          0
Name: count, dtype: int64

=== Datumbereik ===
Min date_only : 2018-12-31
Max date_only : 2026-02-

## Stap 3 — Tijdreeksvolledigheid: verwachte vs. werkelijke observaties

Voor een correcte tijdreeksanalyse en het berekenen van lag-features is het 
essentieel dat de tijdreeks per parking geen onverwachte gaten heeft. 

**Verwacht**: elke parking heeft voor elke dag × 24 uur = 24 observaties.
Totaal verwacht per parking = aantal actieve dagen × 24.

Gaten in de tijdreeks zijn problematisch voor:
- Autocorrelatie-gebaseerde modellen (SARIMA, LSTM)
- Lag-features (lag_1h, lag_24h, lag_168h)
- Rolling-gemiddelden

We berekenen de completeness ratio per parking per jaar.


In [7]:
df_completeness = compute_shortterm_completeness(df_st)

print("=== Tijdreeksvolledigheid per parking per jaar ===")
print(df_completeness.to_string(index=False))

low_coverage = df_completeness[df_completeness["completeness"] < 0.95]
log(
    check_name="Tijdreeksvolledigheid (<95%)",
    dataset="shortterm",
    result=f"{len(low_coverage)} parking-jaar combinaties onder 95%",
    n_affected=int(low_coverage["missing_obs"].sum()),
    action="Gedocumenteerd; geen verwijdering - periodes worden geflagd",
)



=== Tijdreeksvolledigheid per parking per jaar ===
   parking_id  year  n_days  expected_obs  actual_obs  missing_obs  completeness
      P Bruul  2018       1            24           1           23        0.0417
      P Bruul  2019     363          8712        8645           67        0.9923
      P Bruul  2020     366          8784        8767           17        0.9981
      P Bruul  2021     282          6768        6123          645        0.9047
      P Bruul  2022     166          3984        3189          795        0.8005
      P Bruul  2023     161          3864         911         2953        0.2358
      P Bruul  2024     128          3072       14960       -11888        4.8698
      P Bruul  2025     365          8760       35119       -26359        4.0090
      P Bruul  2026      32           768        3054        -2286        3.9766
P Grote Markt  2018       1            24           1           23        0.0417
P Grote Markt  2019     361          8664        5649     

## Stap 4 — Duplicatendetectie

Een duplicaat in de tijdreeks is gedefinieerd als een rij met dezelfde 
combinatie van `(parking_id, rounded_hour)`. Het schema toont 54.436 unieke 
`rounded_hour` waarden tegenover 697.218 rijen en 10 parkings → 
theoretisch maximum is 54.436 × 10 = 544.360 unieke combinaties. 
De 697.218 rijen suggereren mogelijke duplicaten of meerdere observaties 
per uurslot (bijv. correctie-records).


In [8]:
df_st, n_exact_dupes, n_func_dupes = deduplicate_shortterm(df_st)

log(
    "Exacte duplicaten",
    "shortterm",
    f"{n_exact_dupes} rijen",
    n_exact_dupes,
    "Verwijderen" if n_exact_dupes > 0 else "Geen actie",
)
log(
    "Functionele duplicaten (parking_id + rounded_hour)",
    "shortterm",
    f"{n_func_dupes} rijen",
    n_func_dupes,
    "Laatste record bewaren (meest recente publicatie)" if n_func_dupes > 0 else "Geen actie",
)

print(f"Exacte duplicaten: {n_exact_dupes}")
print(f"Functionele duplicaten: {n_func_dupes}")
print(f"Nieuwe shape na deduplicatie: {df_st.shape}")



Exacte duplicaten: 0
Functionele duplicaten: 412694
Nieuwe shape na deduplicatie: (284524, 18)


## Stap 5 — Bezettingswaarden: fysische plausibiliteit

Op basis van het schema is `occupancy_rate` reeds begrensd [0, 1] in de shortterm 
data. Toch controleren we:

1. **Interne consistentie**: is `occupancy_rate ≈ occupied / override`?  
   Tolerantie: |berekende rate - opgeslagen rate| > 0.01 wijst op inconsistentie.
2. **Negatieve waarden**: `number_of_occupied_spaces < 0` is fysisch onmogelijk.
3. **Bezetting > override**: `occupied > number_of_spaces_override`.
4. **Frozen sensor**: ≥ 4 opeenvolgende identieke bezettingswaarden per parking 
   (Gao et al., 2023 hanteren een vergelijkbaar criterium voor sensor-uitval).

Voor de longterm-data is de occupancy_rate-kolom deels NULL (zie schema) — 
dit vereist aparte afhandeling.


In [9]:
df_st, n_inconsistent, n_overcapacity, n_negative = apply_shortterm_quality_flags(df_st)

log(
    "Occupancy rate inconsistentie (>0.01)",
    "shortterm",
    f"{n_inconsistent} rijen ({n_inconsistent / len(df_st) * 100:.2f}%)",
    n_inconsistent,
    "Flaggen als 'occ_rate_inconsistent'",
)
log(
    "Bezetting > capaciteitsoverride",
    "shortterm",
    f"{n_overcapacity} rijen",
    n_overcapacity,
    "Flaggen als 'flag_overcapacity'",
)
log(
    "Negatieve bezetting",
    "shortterm",
    f"{n_negative} rijen",
    n_negative,
    "Verwijderen" if n_negative > 0 else "Geen actie",
)

print(f"\nFlag-verdeling:\n{df_st[['flag_occ_inconsistent', 'flag_overcapacity']].sum()}")




Flag-verdeling:
flag_occ_inconsistent    0
flag_overcapacity        0
dtype: int64


In [10]:
df_st, n_frozen = add_frozen_sensor_flag(df_st, freeze_threshold=FREEZE_THRESHOLD)

log(
    "Frozen sensor (>=4 identieke opeenvolgende uren)",
    "shortterm",
    f"{n_frozen} rijen ({n_frozen / len(df_st) * 100:.2f}%)",
    n_frozen,
    "Flaggen als 'flag_frozen_sensor'; niet verwijderd",
)

print("\nFrozen sensor rijen per parking:")
print(df_st.groupby("parking_id")["flag_frozen_sensor"].sum().sort_values(ascending=False))




Frozen sensor rijen per parking:
parking_id
P Bruul          7099
P Maarten        5780
P Komet          3338
P Lamot          2356
P Tinel          1648
P Kathedraal     1154
P Hoogstraat     1050
P Keerdok         868
P Veemarkt        658
P Grote Markt     441
Name: flag_frozen_sensor, dtype: int64


## Stap 6 — Zomertijdcheck (DST)

België schakelt twee keer per jaar om:
- **Zomertijd**: laatste zondag van maart (klok +1u → uur 02:00 ontbreekt)
- **Wintertijd**: laatste zondag van oktober (klok -1u → uur 02:00 dubbelzinnig)

`publication_time` is TIMESTAMP_NS (timezone-naïef), `created_at` heeft wel 
timezone. Rond de DST-overgangen verwachten we anomalieën in de tijdreeks:
- Ontbrekend uur in maart → gap van 2 uur
- Dubbel uur in oktober → mogelijke duplicaat of sprong

We detecteren dit door per parking per dag te controleren of er 
exact 24 unieke uurwaarden aanwezig zijn.


In [11]:
dst_check, n_dst_anom = check_dst_hour_anomalies(df_st)

print("=== Uurtellingen op DST-overgangsdagen ===")
if len(dst_check) > 0:
    print(dst_check.to_string(index=False))
else:
    print("Geen DST-datums aanwezig in data of alle uren compleet.")

log(
    "DST-uuranomalieen",
    "shortterm",
    f"{n_dst_anom} parking-dag combinaties",
    n_dst_anom,
    "Gedocumenteerd; geen verwijdering",
)



=== Uurtellingen op DST-overgangsdagen ===
   parking_id  date_only  n_unique_hours
      P Bruul 2019-03-31              24
      P Bruul 2019-10-27              23
      P Bruul 2020-03-29              24
      P Bruul 2020-10-25              23
      P Bruul 2021-03-28              24
      P Bruul 2022-10-30              20
      P Bruul 2024-10-27              24
      P Bruul 2025-03-30              24
      P Bruul 2025-10-26              24
P Grote Markt 2019-03-31              23
P Grote Markt 2019-10-27              22
P Grote Markt 2020-03-29              19
P Grote Markt 2020-10-25              18
P Grote Markt 2022-10-30              21
P Grote Markt 2023-03-26              24
P Grote Markt 2023-10-29              21
P Grote Markt 2024-03-31              24
P Grote Markt 2024-10-27              24
P Grote Markt 2025-03-30              24
P Grote Markt 2025-10-26              24
 P Hoogstraat 2023-03-26              19
 P Hoogstraat 2023-10-29              19
 P Hoogstraat 

## Stap 7 — `timeseries_longterm.parquet` kwaliteitscheck

De longterm-data heeft een aantal specifieke kenmerken:
- **7 parkings** (subset van de 10 in shortterm — niet alle parkings hebben abonnees)
- **Enkel 2024** (year distinct count = 1)
- `number_of_spaces_override`: min=20, max=100 → veel kleiner dan shortterm 
  (abonneesplaatsen zijn een gereserveerde subset van de totale capaciteit)
- `occupancy_rate`: heeft NULL-waarden (zie schema)
- `number_of_occupied_spaces`: min=1, max=64

Conceptueel onderscheid (belangrijk voor modellering):
- Shortterm: fluctuerende vraag, sterk afhankelijk van externe factoren 
  (weer, evenementen, vakantie) → hoofddataset voor RQ1
- Longterm: structurele abonneesbezetting, meer stabiel en inertiegedreven → 
  supplementaire analyse of aparte modelleringsdoelstelling


In [12]:
df_lt = read_longterm_timeseries(PROJECT_ROOT)

print(f"Shape         : {df_lt.shape}")
print(f"\nNull-tellingen:\n{df_lt.isnull().sum()}")
print(f"\nJaar(en)      : {df_lt['year'].unique()}")
print(f"\nParkings ({df_lt['parking_id'].nunique()}):")
print(df_lt["parking_id"].value_counts())

print(f"\nnumber_of_spaces_override statistieken:")
print(df_lt["number_of_spaces_override"].describe())

print(f"\nnumber_of_occupied_spaces statistieken:")
print(df_lt["number_of_occupied_spaces"].describe())



Shape         : (46643, 18)

Null-tellingen:
parking_id                   0
parking_id_hash              0
parking_type                 0
kind                         0
publication_time             0
created_at                   0
year                         0
month                        0
date_only                    0
date_with_day                0
rounded_hour                 0
hour                         0
weekday_int                  0
weekday_name                 0
day_type                     0
number_of_spaces_override    0
number_of_occupied_spaces    0
occupancy_rate               0
dtype: int64

Jaar(en)      : [2024]

Parkings (7):
parking_id
P Keerdok                            8750
P Kathedraal                         8727
P Hoogstraat                         8483
P Grote Markt                        8471
P Komet                              5962
P Maarten                            5948
P Tinel                               302
Bike Halte Mechelen-Veemarkt            

In [13]:
df_lt, lt_stats = process_longterm_quality(df_lt, freeze_threshold=FREEZE_THRESHOLD)

log(
    "Null occupancy_rate",
    "longterm",
    f"{lt_stats['n_null_occ']} nulls",
    lt_stats["n_null_occ"],
    "Herberekenen als occupied/override",
)
log(
    "Occupancy inconsistentie",
    "longterm",
    f"{lt_stats['n_incons_lt']} rijen",
    lt_stats["n_incons_lt"],
    "Flaggen",
)
log(
    "Frozen sensor longterm",
    "longterm",
    f"{lt_stats['n_frozen_lt']} rijen",
    lt_stats["n_frozen_lt"],
    "Flaggen",
)
log(
    "Functionele duplicaten",
    "longterm",
    f"{lt_stats['n_dupes_lt']}",
    lt_stats["n_dupes_lt"],
    "Verwijderen" if lt_stats["n_dupes_lt"] > 0 else "Geen actie",
)

print(f"Null occupancy_rate voor imputatie: {lt_stats['n_null_occ']}")
print(f"Occupancy inconsistentieen: {lt_stats['n_incons_lt']}")
print(f"Frozen sensor rijen: {lt_stats['n_frozen_lt']}")
print(f"Functionele duplicaten: {lt_stats['n_dupes_lt']}")



Null occupancy_rate voor imputatie: 0
Occupancy inconsistentieen: 0
Frozen sensor rijen: 4112
Functionele duplicaten: 0


## Stap 8 — Capaciteitsoverride vs. locatiedata kruischeck

`number_of_spaces_override` in de tijdreeks is de **operationele** capaciteit 
op dat uurmoment. `total_capacity` in `parking_location` is de **totale** 
gecertificeerde capaciteit.

Relatie: `override ≤ total_capacity` altijd.  
Uitzondering: tijdelijke capaciteitsuitbreidingen of data-invoerfouten.

We kruisen beide om parkings te identificeren waar de override-waarde de 
gecertificeerde capaciteit overstijgt.


In [14]:
cap_check, n_exceeds = crosscheck_override_capacity(df_st, df_loc)

print("=== Capaciteitskruischeck (shortterm max override vs. total_capacity) ===")
print(cap_check.to_string(index=False))

log(
    "Override > total_capacity",
    "shortterm",
    f"{n_exceeds} parkings",
    n_exceeds,
    "Documenteren; mogelijke dynamische capaciteitswijziging of datafout",
)



=== Capaciteitskruischeck (shortterm max override vs. total_capacity) ===
   parking_id  max_override  total_capacity  override_exceeds_capacity
      P Bruul         350.0           350.0                      False
P Grote Markt         135.0           155.0                      False
 P Hoogstraat         120.0           109.0                       True
 P Kathedraal          70.0           130.0                      False
    P Keerdok         416.0           516.0                      False
      P Komet          60.0           124.0                      False
      P Lamot         235.0           255.0                      False
    P Maarten         189.0           189.0                      False
      P Tinel          90.0           124.0                      False
   P Veemarkt          99.0           129.0                      False


## Stap 9 — Overzicht van alle kwaliteitsflags

De gecleande datasets bevatten de volgende boolean-flags (0/1):

| Flag | Betekenis |
|---|---|
| `flag_occ_inconsistent` | `\|occupancy_rate - occupied/override\| > 0.01` |
| `flag_overcapacity` | `occupied > override` (shortterm only) |
| `flag_frozen_sensor` | ≥ 4 opeenvolgende identieke bezettingswaarden |

**Beleid voor modellering**: rijen met `flag_frozen_sensor = 1` worden 
**uitgesloten** van modeltraining (niet verwijderd uit dataset). 
Rijen met `flag_occ_inconsistent = 1` worden bewaard maar de 
`occupancy_rate` wordt herberekend vanuit `occupied/override`.


In [15]:
df_st, n_fix_st = recompute_occupancy_for_inconsistent(df_st)
df_lt, n_fix_lt = recompute_occupancy_for_inconsistent(df_lt)

print(f"[shortterm] {n_fix_st} occupancy_rate waarden gecorrigeerd via herberekening.")
print(f"[longterm] {n_fix_lt} occupancy_rate waarden gecorrigeerd via herberekening.")



[shortterm] 0 occupancy_rate waarden gecorrigeerd via herberekening.
[longterm] 0 occupancy_rate waarden gecorrigeerd via herberekening.


In [16]:
print("=== FLAG SAMENVATTING SHORTTERM ===")
flag_cols_st = [c for c in df_st.columns if c.startswith("flag_")]
print(df_st[flag_cols_st].sum().to_frame("n_flagged"))
print(
    f"\nTotaal bruikbare rijen (geen frozen sensor): {(df_st['flag_frozen_sensor'] == 0).sum():,}"
)

print("\n=== FLAG SAMENVATTING LONGTERM ===")
flag_cols_lt = [c for c in df_lt.columns if c.startswith("flag_")]
print(df_lt[flag_cols_lt].sum().to_frame("n_flagged"))
print(
    f"\nTotaal bruikbare rijen (geen frozen sensor): {(df_lt['flag_frozen_sensor'] == 0).sum():,}"
)



=== FLAG SAMENVATTING SHORTTERM ===
                       n_flagged
flag_occ_inconsistent          0
flag_overcapacity              0
flag_frozen_sensor         24392

Totaal bruikbare rijen (geen frozen sensor): 260,132

=== FLAG SAMENVATTING LONGTERM ===
                       n_flagged
flag_occ_inconsistent          0
flag_frozen_sensor          4112

Totaal bruikbare rijen (geen frozen sensor): 42,531


In [17]:
out_st = resolve_data_path("@data/intermediate/shortterm_cleaned.parquet", PROJECT_ROOT)
df_st.to_parquet(out_st, index=False)
print(f"OK shortterm_cleaned.parquet opgeslagen: {df_st.shape}")

out_lt = resolve_data_path("@data/intermediate/longterm_cleaned.parquet", PROJECT_ROOT)
df_lt.to_parquet(out_lt, index=False)
print(f"OK longterm_cleaned.parquet opgeslagen:  {df_lt.shape}")



OK shortterm_cleaned.parquet opgeslagen: (284524, 21)
OK longterm_cleaned.parquet opgeslagen:  (46643, 20)


In [18]:
df_quality = qlogger.to_frame()

out_qr = resolve_data_path("@data/intermediate/quality_report.csv", PROJECT_ROOT)
df_quality.to_csv(out_qr, index=False)

print("=== KWALITEITSRAPPORT ===")
print(df_quality.to_string(index=False))
print(f"\nOK quality_report.csv opgeslagen: {out_qr}")



=== KWALITEITSRAPPORT ===
                                             check          dataset                                result  n_affected                                                              action
                                   Aantal parkings parking_location                         15 unieke IDs           0                                                                  OK
                      Tijdreeksvolledigheid (<95%)        shortterm 33 parking-jaar combinaties onder 95%       48360         Gedocumenteerd; geen verwijdering - periodes worden geflagd
                                 Exacte duplicaten        shortterm                               0 rijen           0                                                          Geen actie
Functionele duplicaten (parking_id + rounded_hour)        shortterm                          412694 rijen      412694                   Laatste record bewaren (meest recente publicatie)
             Occupancy rate inconsistentie (

## Samenvatting

| Output | Inhoud |
|---|---|
| `shortterm_cleaned.parquet` | 697.218 rijen + 3 flag-kolommen, duplicaten verwijderd, occupancy gecorrigeerd |
| `longterm_cleaned.parquet`  | 46.643 rijen + 2 flag-kolommen, null occupancy geïmputeerd |
| `quality_report.csv`        | Volledig audittrail van alle datakwaliteitschecks |

### Beslissingen voor modellering (gedocumenteerd)
- Rijen met `flag_frozen_sensor = 1` worden **geëxcludeerd** bij feature engineering
- Occupancy_rate wordt **altijd herberekend** vanuit `occupied/override` als 
  grondwaarheid (numeriek stabieler dan de opgeslagen waarde)
- Jaar 2018 in shortterm: slechts gedeeltelijke data → **uitsluiten** voor 
  modellen die volledigheid per jaar vereisen

### Volgende stap
➡️ `03_weather_cleaning.ipynb` — Laden en verwerken van `aws_1hour-5.csv`: 
timezone-correctie, missing value imputatie, feature-selectie voor weerdata.

---
*Referenties:*  
- Gao et al. (2023). ConvGRU with wavelet denoising for parking prediction.  
- Nezhadettehad et al. (2025). Bayesian NN for uncertainty-aware parking forecasting.  
- Tanui et al. (2025). ROLS-enhanced MLR with SHAP for occupancy prediction.  
- Wan et al. (2023). Attention-enhanced TCN with extreme weather awareness.
